# Bolsa de Valores — demonstração temporal

Ativos e cotações integralmente fictícios. Reconstrução pública nova inspirada no contexto do painel original; não é recuperação dos dados, DAX ou PBIX antigos. Não há sinal de negociação ou retorno comercial comprovado.

In [ ]:
from pathlib import Path
import os
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/bolsa').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from bolsa.synthetic import generate
from bolsa.contracts import validate
from bolsa.analytics import analyze, narrative
from bolsa.pipeline import publish


## Contrato e calendário
Uma cotação por ativo/sessão; seis ativos DEMO. Calendário de segunda a sexta inclui possíveis feriados reais, portanto não representa sessões NASDAQ. Moeda USD em valores fictícios. Ajuste somente por split, sem dividendos, custos ou impostos.

In [ ]:
dataset = validate(generate())
assert len(dataset.quotes) == len(dataset.assets) * len(dataset.metadata['sessions'])
print({'assets': len(dataset.assets), 'sessions': len(dataset.metadata['sessions']), 'quotes': len(dataset.quotes), 'synthetic': True})
dataset.quotes.head()


## Retornos e janelas
Retorno diário por ativo usa preço ajustado. Primeiro retorno ausente, base 100 no primeiro preço; volatilidade móvel exige 20 retornos completos, usa ddof=1 e convenção anual de 252. Momentum usa 21 observações de defasagem; drawdown usa o máximo acumulado desde o início da janela. Nenhuma janela atravessa ativos.

In [ ]:
analysis = analyze(dataset)
assert analysis.daily.groupby("symbol").head(1).return_daily.isna().all()
analysis.summary


## Duas variações mensais diferentes
Retorno mensal de preço compara últimos preços ajustados de meses consecutivos. Variação da média mensal bruta compara médias de fechamento: não é retorno e é sensível a splits. Primeiro mês ou mês anterior ausente não vira zero. Completude é relativa à declaração de cobertura, não certificação do calendário de uma bolsa real.

In [ ]:
analysis.monthly[['symbol', 'month', 'return_monthly_price', 'mean_close_mom', 'period_complete_declared']].head(12)


## Desdobramento fictício
DEMO_A possui split 2:1. O preço bruto reduz mecanicamente; a série ajustada mantém a evolução do processo fictício. Volume continua em ações brutas e não mede spread, profundidade ou liquidez completa.

In [ ]:
event = dataset.metadata['split_demo']
rows = analysis.daily.query('symbol == @event["symbol"]')
position = rows.index[rows.date.eq(event['date'])][0]
analysis.daily.loc[position-1:position+1, ['date', 'close', 'adjusted_close', 'return_daily']]


## Publicação e SQL real local
Cada geração contém SQLite com dimensões de ativo/data e fato de cotação. Chaves, restrições e carga transacional são verificadas; consultas mensais SQL são reconciliadas com Python antes de confirmar latest.json. Não há dependência de servidor PostgreSQL.

In [ ]:
output = Path(os.environ.get('BOLSA_NOTEBOOK_OUTPUT', str(ROOT / 'outputs/notebook')))
manifest = publish(dataset, output)
from IPython.display import Image, display
for name in ('painel.png', 'retornos_mensais.png'):
    display(Image(filename=str(output / manifest['directory'] / name)))
assert manifest['synthetic'] is True


## Storytelling descritivo
Texto determinístico gerado a partir dos valores computados, sem alegar execução de Smart Narrative do Power BI. Resultados são descritivos, sem recomendação ou previsão.

In [ ]:
print(narrative(analysis))


## Limites
Os dados não representam desempenho de empresas reais. Não há correção por dividendos, backtest, conversão cambial, validação DAX ou Power BI Desktop. A cobertura do gerador não é o calendário oficial de mercado. A completude de dados externos dependeria da fonte e das sessões declaradas. Consulte docs/METODOLOGIA.md e docs/VALIDACAO.md.